# Operating-state classification with OYICD

Operating-state classification identifies how a machine is running rather than
whether it has failed. OYICD encodes one of eight modes in every recording name.

## Learning goals

- discover downloaded recordings without hard-coded paths;
- summarize each recording with Polars;
- classify modes while holding out later recordings.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdmdata").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import pdmdata
from pdmdata.oyicd import inventory, load
from pdmdata.oyicd.loader import SENSOR_COLUMNS
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score

In [ ]:
pdmdata.download("oyicd")
files = inventory().sort("filename")
print(f"Unique recordings: {files.height}")

def summarize_recording(filename):
    frame = load(filename)
    return frame.select(
        *[pl.col(column).mean().alias(f"{column}_mean") for column in SENSOR_COLUMNS],
        *[pl.col(column).std().alias(f"{column}_std") for column in SENSOR_COLUMNS],
    ).with_columns(
        pl.lit(filename).alias("recording"),
        pl.lit(frame["mode"][0]).alias("mode"),
    )

recordings = pl.concat([summarize_recording(name) for name in files["filename"]])
recordings.group_by("mode").len().sort("mode")

In [ ]:
recordings = recordings.with_columns(
    pl.int_range(pl.len()).over("mode").alias("mode_order"),
    pl.len().over("mode").alias("mode_count"),
)
train = recordings.filter(pl.col("mode_order") < pl.col("mode_count") * 0.8)
test = recordings.filter(pl.col("mode_order") >= pl.col("mode_count") * 0.8)
predictors = [
    column for column, dtype in recordings.schema.items()
    if dtype.is_numeric() and column not in {"mode", "mode_order", "mode_count"}
]

classifier = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=0,
)
classifier.fit(train.select(predictors).fill_null(0).to_numpy(), train["mode"].to_numpy())
prediction = classifier.predict(test.select(predictors).fill_null(0).to_numpy())
print(
    "Later-recording balanced accuracy:",
    f"{balanced_accuracy_score(test['mode'].to_numpy(), prediction):.3f}",
)

## Interpretation and limitations

Mode is an operating condition, not a fault label. The dataset loader validates
and counts duplicate source copies once. Features use only the eight sensors;
time and mode columns are excluded. The split holds out later recordings within
each mode; it is not a single global time cutoff across all modes. A production model should also test whether mode recognition transfers
between machines and component ages.